In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:

dfd = pd.read_csv(r"C:\Users\rishy\Desktop\New folder\infosys\day.csv")

In [ ]:
print(f"Initial Shape: {dfd.shape}")
dfd.drop_duplicates(inplace=True)
dfd.dropna(inplace=True)

def remove_outliers(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        df = df[(df[col] >= (Q1 - 1.5 * IQR)) & (df[col] <= (Q3 + 1.5 * IQR))]
    return df

numeric_features = ['temp', 'atemp', 'hum', 'windspeed']
dfd = remove_outliers(dfd, numeric_features + ['cnt'])

Initial Shape: (731, 16)


In [19]:
X = dfd.drop(columns=['instant', 'dteday', 'casual', 'registered', 'cnt'])
y = dfd['cnt']

In [20]:
cat_features = ['season', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'yr']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', numeric_features)
    ])

In [21]:
pipeline = Pipeline([
    ('prep', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [15, 20, None],
    'regressor__min_samples_split': [2, 5]
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'regressor__max_depth': [15, 20, ...], 'regressor__min_samples_split': [2, 5], 'regressor__n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('num', ...)]"


In [23]:
train_acc = grid_search.score(X_train, y_train) * 100
test_acc = grid_search.score(X_test, y_test) * 100

print(f"\n--- Performance Results ---")
print(f"Train Accuracy (R2): {train_acc:.2f}%")
print(f"Test Accuracy (R2): {test_acc:.2f}%")




--- Performance Results ---
Train Accuracy (R2): 98.14%
Test Accuracy (R2): 88.53%
